# Notebook 0: Plot Data 

This is the first notebook in the sequence for the HRS Botany project.

In it we will download, prepare, and merge plot data from the UCSC Forest Ecology Research Plot and the Open Forest Observatory.

The plot data just includes the plot metadata for the FERP plot and all of the OFO plots. We will use the merged plot dataset to determine which EnMAP images to download in a subsequent notebook. 

__Notebook Inputs:__
- OFO Plots Data (_ofo_ground-reference_plots.gpkg_)
- FERP Data (_FERP123merged_20231029.csv_)


__Notebook Outputs:__
- Plot Dataset (_data/plots.csv_)


In [95]:
# Imports

import numpy as np
import pandas as pd

import geopandas as gpd
from shapely.geometry import Point

from hrs_botany.data_utils import clean_ofo_survey_date, load_ferp_species_table

## 1. OFO Data Acquisition

First we will prepare the OFO plot data. 

The OFO plot data is separate from the trees data, and has a single row for each plot. Each plot row includes many metadata fields, but for our purposes, the columns of interest are _plot_id_, _plot_area_ha_, _survey_date_, and 
_geometry_. 


In [96]:
ofo_plots_df = gpd.read_file('../../data/ofo/ofo_ground-reference_plots.gpkg')

In [97]:
ofo_plots_df = ofo_plots_df[['plot_id', 'plot_area_ha', 'survey_date', 'geometry']]

ofo_plots_df.head(5)

,plot_id,plot_area_ha,survey_date,geometry
0,0070,4.273060,2008.0,"MULTIPOLYGON (((-120.00103 38.17605, -120.0011..."
1,0081,4.000000,201910.0,"POLYGON ((-116.76897 33.81007, -116.76767 33.8..."
2,0083,4.003663,202008.0,"MULTIPOLYGON (((-123.86914 40.77879, -123.8691..."
3,0084,4.000000,201907.0,"MULTIPOLYGON (((-119.73617 37.09988, -119.7361..."
4,0087,3.979825,201906.0,"MULTIPOLYGON (((-119.02333 36.96424, -119.0210..."


In [98]:
ofo_plots_df.plot_area_ha.describe()

count    258.000000
mean       0.449594
std        0.864867
min        0.020106
25%        0.079862
50%        0.129533
75%        0.291864
max        4.273060
Name: plot_area_ha, dtype: float64

It is important to our study to restrict our plot selection to plots of a large enough area to cover several contiguous EnMAP pixels (30x30m). Otherwise, we will not be able to determine species absence in a full pixel. 

__We choose a minimum plot size of 0.5 hectare – this value needs a stronger scientific rationale, and should be revisited.__

In [99]:
ofo_plots_df = ofo_plots_df[ofo_plots_df.plot_area_ha > 0.5].reset_index(drop=True)

ofo_plots_df = ofo_plots_df[ofo_plots_df['plot_id'].notna()].reset_index(drop=True)

ofo_plots_df = ofo_plots_df.sort_values(by='plot_id').reset_index(drop=True)    

In [100]:
print('Number of OFO plots:', len(ofo_plots_df))

ofo_plots_df.plot_area_ha.describe()


Number of OFO plots: 33


count    33.000000
mean      2.414287
std       1.160409
min       0.785000
25%       1.393045
50%       2.522393
75%       3.459541
max       4.273060
Name: plot_area_ha, dtype: float64

In [101]:
# Now let's fix the survey date format in the OFO plots dataframe.

ofo_plots_df["survey_date"] = clean_ofo_survey_date(ofo_plots_df["survey_date"])



In [102]:
ofo_plots_df.head(5)

,plot_id,plot_area_ha,survey_date,geometry
0,0058,1.463947,2020-08-09,"POLYGON ((-121.11482 40.12925, -121.11517 40.1..."
1,0059,0.945688,2020-07-15,"POLYGON ((-122.68571 38.81993, -122.68595 38.8..."
2,0060,1.308436,2020-08-23,"POLYGON ((-123.56877 40.29993, -123.56915 40.2..."
3,0061,1.286581,2020-07-04,"POLYGON ((-122.5369 41.03392, -122.53713 41.03..."
4,0062,1.398960,2021-06-21,"POLYGON ((-123.56378 40.30005, -123.56385 40.3..."


We will now make a copy of the ofo plot df that will serve as the overall plot df. We will merge FERP data later.

In [103]:
plots_df = ofo_plots_df.copy()

## 2. FERP Data Acquisition

Now we will prepare the FERP plot.

FERP is a single plot, but it has been censused by multiple surveys. We will include a row for each of these surveys.

Since FERP is just a single plot, it will just be a single row in our plot dataset.

I could not find a polygon file for the FERP plot, so we're doing it manually.

We'll load the FERP dataset, and create a rotated rectangular boundary for the plot geometry.

In [104]:
ferp_path = '../../data/ferp/geoforest/doi_10_5061_dryad_6q573n64s__v20240129/FERP123merged_20231029.csv'

# Load the dataset
ferp_plot_df = pd.read_csv(ferp_path)

/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_17971/3633041921.py:4: DtypeWarning: Columns (13,16,17,23,25,27,32) have mixed types. Specify dtype option on import or set low_memory=False.
  ferp_plot_df = pd.read_csv(ferp_path)


In [105]:
ferp_plot_df.columns

Index(['quadrat', 'tag', 'stemtag', 'stemtag1', 'code6', 'east_m', 'north_m',
       'east_UTM', 'north_UTM', 'dsh1_mm', 'dsh2_mm', 'dsh3_mm', 'dsh1m_mm',
       'date1', 'date2', 'date3', 'status1', 'condition1', 'status2',
       'condition2', 'status3', 'condition3', 'first_census', 'irreg_dsh',
       'hom_m', 'multi1', 'stems1', 'multi2', 'multi3', 'basalarea1_m2',
       'code6fix', 'locfix', 'notes2', 'notes3'],
      dtype='object')

In [106]:
ferp_plot_df = ferp_plot_df[['east_UTM', 'north_UTM', 'date1', 'date2', 'date3']]

In [107]:


# Create GeoDataFrame
df_geo = ferp_plot_df.dropna(subset=["east_UTM", "north_UTM"]).copy()
gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(df_geo["east_UTM"], df_geo["north_UTM"]),
    crs="EPSG:32610"  # UTM Zone 10N
)

# Create rotated bounding box from stem points
ferp_boundary = gpd.GeoDataFrame(
    geometry=[gdf.unary_union.minimum_rotated_rectangle],
    crs=gdf.crs  # same CRS as stem points
)

/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_17971/884955862.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[gdf.unary_union.minimum_rotated_rectangle],


In [108]:
ferp_boundary.iloc[0]

geometry    POLYGON ((582305.294 4096649.796, 582406.551 4...
Name: 0, dtype: geometry

In [109]:
import geopandas as gpd

# — assume you already have:
#    • ferp_boundary: a GeoDataFrame in EPSG:32610 (your minimum_rotated_rectangle)
#    • ofo_plots_df: a GeoDataFrame whose .crs is EPSG:4326 (lat/lon) or similar

# 1. Check and align CRSes
print("FERP CRS:", ferp_boundary.crs)
print("OFO CRS:", ofo_plots_df.crs)

# if ofo_plots_df.crs is None, set it explicitly:
# ofo_plots_df = ofo_plots_df.set_crs("EPSG:4326", inplace=False)

# 2. Reproject the FERP rectangle into the OFO CRS
ferp_boundary_ofo = ferp_boundary.to_crs(ofo_plots_df.crs).geometry.values[0]
ferp_area_ha = ferp_boundary.geometry.area.iloc[0] / 10_000



FERP CRS: EPSG:32610
OFO CRS: EPSG:4326


We will now manually add three rows to the plots_df for each of the FERP surveys. Each will have a unique plot_id, it will have the average survey date, and will have the same FERP boundary.

The date logic needs to be refined.

In [110]:
ferp_plots_df = pd.DataFrame({
    'plot_id': ['f1', 'f2', 'f3'],
    'plot_area_ha': [ferp_area_ha, ferp_area_ha, ferp_area_ha],
    'survey_date': [pd.to_datetime(ferp_plot_df.date1).mean(), pd.to_datetime(ferp_plot_df.date2).mean(), pd.to_datetime(ferp_plot_df.date3).mean()],
    'geometry': [ferp_boundary_ofo, ferp_boundary_ofo, ferp_boundary_ofo]
})

/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_17971/2258156113.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  'survey_date': [pd.to_datetime(ferp_plot_df.date1).mean(), pd.to_datetime(ferp_plot_df.date2).mean(), pd.to_datetime(ferp_plot_df.date3).mean()],
/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_17971/2258156113.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  'survey_date': [pd.to_datetime(ferp_plot_df.date1).mean(), pd.to_datetime(ferp_plot_df.date2).mean(), pd.to_datetime(ferp_plot_df.date3).mean()],
/var/folders/qx/bpj16cl90cq20swjd4j79cdh0000gn/T/ipykernel_17971/2258156113.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling 

In [111]:
ferp_plots_df

,plot_id,plot_area_ha,survey_date,geometry
0,f1,15.99267,2007-05-21 23:52:57.125193216,POLYGON ((-122.07483338054512 37.0124163255955...
1,f2,15.99267,2013-05-31 08:36:15.501506560,POLYGON ((-122.07483338054512 37.0124163255955...
2,f3,15.99267,2019-02-10 17:20:12.165758464,POLYGON ((-122.07483338054512 37.0124163255955...


We will now join the ferp plots to the overall plots df

In [112]:
plots_df = pd.concat([plots_df, ferp_plots_df], ignore_index=True)

In [113]:
plots_df

,plot_id,plot_area_ha,survey_date,geometry
0,0058,1.463947,2020-08-09 00:00:00.000000000,"POLYGON ((-121.11482 40.12925, -121.11517 40.1..."
1,0059,0.945688,2020-07-15 00:00:00.000000000,"POLYGON ((-122.68571 38.81993, -122.68595 38.8..."
2,0060,1.308436,2020-08-23 00:00:00.000000000,"POLYGON ((-123.56877 40.29993, -123.56915 40.2..."
3,0061,1.286581,2020-07-04 00:00:00.000000000,"POLYGON ((-122.5369 41.03392, -122.53713 41.03..."
4,0062,1.398960,2021-06-21 00:00:00.000000000,"POLYGON ((-123.56378 40.30005, -123.56385 40.3..."
5,0063,1.462966,2020-07-25 00:00:00.000000000,"POLYGON ((-122.5425 41.02278, -122.54262 41.02..."
6,0066,1.404556,2021-08-29 00:00:00.000000000,"POLYGON ((-119.42135 37.23617, -119.42204 37.2..."
7,0067,1.393045,2021-08-31 00:00:00.000000000,"POLYGON ((-119.41176 37.26544, -119.41177 37.2..."
8,0068,3.230183,2019-09-15 00:00:00.000000000,"POLYGON ((-120.08843 38.96572, -120.08844 38.9..."
9,0069,3.459541,2016-01-01 00:00:00.000000000,"MULTIPOLYGON (((-120.01534 38.18483, -120.0177..."


In [114]:
plots_df.to_csv('./data/plots.csv')